# IoU and Dice versus box-prompt perturbation magnitude

This notebook relates segmentation performance to the geometric difference between each perturbed prompt box and its original box. It is separate from the earlier robustness-analysis notebook.

The perturbation magnitude is defined as:

$$\text{box non-overlap (\%)} = 100 \times (1 - \operatorname{IoU}(B_{original}, B_{perturbed})).$$

Thus, 0% means that the boxes are identical, while a larger value means a stronger perturbation. In all-image plots, the dashed horizontal reference is the mean original-prompt score over all images. In class-specific plots, it is the mean original-prompt score for that class.

In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"

## Configuration

In [ ]:
RESULTS_JSONL = Path("./results/sam3_coco_box_robustness.jsonl")
FIGURE_DIR = Path("./robustness-plots-box-perturbation")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

N_ALL_IMAGE_BINS = 10
N_CLASS_BINS = 5
FACET_COLUMNS = 3
SCATTER_ALPHA_ALL = 0.16
SCATTER_ALPHA_CLASS = 0.25

## Load the JSONL and calculate box IoU

The resulting long-form table has one row per perturbed prompt. With 100 images and 10 perturbations per image, it contains 1,000 rows.

In [ ]:
def box_iou_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)
    intersection_width = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    intersection_height = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = intersection_width * intersection_height
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else np.nan

def load_prompt_level_results(path: Path):
    prompt_rows = []
    original_rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            original_box = record["original_prompt"]["box_xyxy"]
            original_metrics = record["original_prompt"]["metrics"]
            common = {
                "image_id": record["image_id"],
                "annotation_id": record["annotation_id"],
                "file_name": record["file_name"],
                "category_name": record["category_name"],
            }
            original_rows.append({
                **common,
                "original_iou": float(original_metrics["iou"]),
                "original_dice": float(original_metrics["dice"]),
            })
            for perturbation in record["perturbed_prompts"]:
                perturbed_box = perturbation["box_xyxy"]
                overlap = box_iou_xyxy(original_box, perturbed_box)
                prompt_rows.append({
                    **common,
                    "perturbation_index": perturbation["perturbation_index"],
                    "box_iou": overlap,
                    "box_overlap_percent": 100.0 * overlap,
                    "box_nonoverlap_percent": 100.0 * (1.0 - overlap),
                    "iou": float(perturbation["metrics"]["iou"]),
                    "dice": float(perturbation["metrics"]["dice"]),
                    "original_iou": float(original_metrics["iou"]),
                    "original_dice": float(original_metrics["dice"]),
                })
    return pd.DataFrame(prompt_rows), pd.DataFrame(original_rows)

prompt_df, original_df = load_prompt_level_results(RESULTS_JSONL)
print(f"Loaded {len(prompt_df)} perturbed prompts from {len(original_df)} images")
print(f"Object classes: {original_df['category_name'].nunique()}")
display(prompt_df.head())
display(prompt_df[["box_iou", "box_nonoverlap_percent", "iou", "dice"]].describe())

## Plotting helpers

Faint points show individual perturbed prompts. The solid curve shows the mean score in equal-frequency perturbation bins, with approximate 95% confidence intervals (`1.96 × SEM`). The dashed line is the relevant mean original-prompt score.

In [ ]:
METRICS = {
    "iou": {"label": "IoU", "color": "#4C72B0"},
    "dice": {"label": "Dice", "color": "#DD8452"},
}

def binned_curve(data, x_column, y_column, n_bins):
    working = data[[x_column, y_column]].dropna().copy()
    if working.empty:
        return pd.DataFrame()
    unique_count = working[x_column].nunique()
    q = min(n_bins, unique_count)
    if q < 2:
        return pd.DataFrame({
            "x": [working[x_column].mean()],
            "mean": [working[y_column].mean()],
            "ci95": [1.96 * working[y_column].sem()],
            "n": [len(working)],
        })
    working["bin"] = pd.qcut(working[x_column], q=q, duplicates="drop")
    rows = []
    for _, group in working.groupby("bin", observed=True):
        rows.append({
            "x": group[x_column].mean(),
            "mean": group[y_column].mean(),
            "ci95": 1.96 * group[y_column].sem() if len(group) > 1 else np.nan,
            "n": len(group),
        })
    return pd.DataFrame(rows).sort_values("x")

def draw_score_vs_perturbation(ax, data, metric, original_reference, n_bins, scatter_alpha):
    style = METRICS[metric]
    ax.scatter(
        data["box_nonoverlap_percent"], data[metric],
        s=16, alpha=scatter_alpha, color=style["color"], edgecolors="none",
        label="Perturbed prompts",
    )
    curve = binned_curve(data, "box_nonoverlap_percent", metric, n_bins)
    if not curve.empty:
        ax.errorbar(
            curve["x"], curve["mean"], yerr=curve["ci95"],
            marker="o", linewidth=2.2, capsize=3, color=style["color"],
            label="Binned mean ± 95% CI",
        )
    ax.axhline(
        original_reference, color="black", linestyle="--", linewidth=1.5,
        label=f"Mean original {style['label']}: {original_reference:.3f}",
    )
    ax.set_xlabel("Box perturbation: 100 × (1 − box IoU) [%]")
    ax.set_ylabel(f"Predicted-mask {style['label']}")
    ax.set_ylim(-0.02, 1.02)
    return curve

## 1. All images together

Each panel combines every perturbed prompt from every image. The original-score line is averaged once over images, not over perturbed prompts, so every image contributes equally to the reference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
all_image_curves = {}
for ax, metric in zip(axes, ("iou", "dice")):
    original_reference = original_df[f"original_{metric}"].mean()
    all_image_curves[metric] = draw_score_vs_perturbation(
        ax, prompt_df, metric, original_reference, N_ALL_IMAGE_BINS, SCATTER_ALPHA_ALL
    )
    ax.set_title(f"All images: {METRICS[metric]['label']}")
    ax.legend(fontsize=8)
fig.suptitle("Segmentation performance versus box-prompt perturbation", y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_all_images_scores_vs_box_perturbation.png", dpi=200)
plt.show()

## 2. Separate plots for each object class

One faceted figure is generated for IoU and one for Dice. Every subplot has its own class-specific original-score reference line.

In [ ]:
def plot_classes(metric):
    classes = sorted(prompt_df["category_name"].dropna().unique())
    n_rows = math.ceil(len(classes) / FACET_COLUMNS)
    fig, axes = plt.subplots(
        n_rows, FACET_COLUMNS, figsize=(5.2 * FACET_COLUMNS, 4.1 * n_rows),
        sharex=True, sharey=True, squeeze=False,
    )
    class_curves = {}
    for ax, category in zip(axes.flat, classes):
        class_prompts = prompt_df[prompt_df["category_name"] == category]
        class_original = original_df[original_df["category_name"] == category]
        original_reference = class_original[f"original_{metric}"].mean()
        class_curves[category] = draw_score_vs_perturbation(
            ax, class_prompts, metric, original_reference, N_CLASS_BINS, SCATTER_ALPHA_CLASS
        )
        ax.set_title(
            f"{category} ({len(class_original)} images, {len(class_prompts)} prompts)"
        )
        # Keep one compact legend; the encodings are identical in every facet.
        if category == classes[0]:
            ax.legend(fontsize=7, loc="best")
    for ax in axes.flat[len(classes):]:
        ax.set_visible(False)
    metric_label = METRICS[metric]["label"]
    fig.suptitle(
        f"{metric_label} versus box-prompt perturbation, separately by object class", y=1.01
    )
    fig.tight_layout()
    output = FIGURE_DIR / f"02_class_{metric}_vs_box_perturbation.png"
    fig.savefig(output, dpi=200)
    plt.show()
    return class_curves

class_iou_curves = plot_classes("iou")
class_dice_curves = plot_classes("dice")

## 3. Numerical trend summary by class

For a compact companion table, the following cell estimates an ordinary least-squares slope for score versus one percentage point of box non-overlap. A more negative slope indicates faster performance degradation. This is descriptive and does not account for repeated perturbations from the same image.

In [ ]:
trend_rows = []
for category, group in prompt_df.groupby("category_name"):
    for metric in ("iou", "dice"):
        clean = group[["box_nonoverlap_percent", metric]].dropna()
        slope, intercept = np.polyfit(clean["box_nonoverlap_percent"], clean[metric], 1)
        correlation = clean["box_nonoverlap_percent"].corr(clean[metric])
        trend_rows.append({
            "category_name": category,
            "metric": METRICS[metric]["label"],
            "score_change_per_1pct_nonoverlap": slope,
            "pearson_correlation": correlation,
            "num_prompts": len(clean),
        })
trend_summary = pd.DataFrame(trend_rows)
display(trend_summary.sort_values(["metric", "score_change_per_1pct_nonoverlap"]))
trend_summary.to_csv(FIGURE_DIR / "class_score_vs_perturbation_trends.csv", index=False)